# Binomial option pricing — companion to *Valuation of Options, part 1*

This notebook reproduces the discrete-time pricing examples from the slides (and the accompanying spreadsheet) and lets you experiment with the numbers yourself.

We cover:
1. the **one-period** model priced three equivalent ways — *replication*, *risk-neutral* pricing, and the *pricing kernel*;
2. the **two-period** model priced by **backward induction**;
3. a general **$n$-step** binomial model and its **convergence to the Black–Scholes price** (a bonus / food-for-thought).

Only `numpy` and `matplotlib` are required.


## 0. Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, erf, log, sqrt, exp

def Phi(x):
    """Standard normal CDF (no SciPy needed)."""
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


## 1. One-period model — setup

Same numbers as the slides: today $S_0=125$, $B_0=1$; after one period the bond grows to $B_1=1.02$ ($r=2\%$) and the stock moves to $s_u=150$ ("up") or $s_d=100$ ("down"), with real-world probability $p=0.7$. The claim is a **call with strike $K=50$**.


In [ ]:
S0, B0, B1 = 125.0, 1.0, 1.02
su, sd = 150.0, 100.0
p = 0.7                      # real-world P(up)
K = 50.0
cu, cd = max(su - K, 0.0), max(sd - K, 0.0)   # call payoff in each state
print(f'payoff: c_u = {cu}, c_d = {cd}')


## 2. Method 1 — replication

Find $(\phi,\psi)$ (shares, bonds) reproducing the payoff in **both** states:
$$\phi=\frac{c_u-c_d}{s_u-s_d},\qquad \psi=\frac{c_d s_u - s_d c_u}{B_1(s_u-s_d)},\qquad C_0=\phi S_0+\psi B_0.$$


In [ ]:
phi = (cu - cd) / (su - sd)
psi = (cd * su - sd * cu) / (B1 * (su - sd))
C0_repl = phi * S0 + psi * B0
print(f'phi = {phi:.4f}, psi = {psi:.4f}')
print(f'C0 (replication) = {C0_repl:.4f}')
assert np.isclose(phi*su + psi*B1, cu) and np.isclose(phi*sd + psi*B1, cd)
print('replication check passed')


## 3. Method 2 — risk-neutral pricing

Choose $q$ so that $S/B$ is a martingale, then discount the expected payoff:
$$q=\frac{B_1/B_0-s_d/S_0}{s_u/S_0-s_d/S_0},\qquad C_0=\frac{1}{B_1}\,\mathbb{E}_{\mathbb{Q}}[C_1].$$


In [ ]:
q = (B1/B0 - sd/S0) / (su/S0 - sd/S0)
C0_rn = (q*cu + (1-q)*cd) / B1
print(f'q = {q:.4f}')
print(f'C0 (risk-neutral) = {C0_rn:.4f}')
print(f'E_P[S1]/B1 = {(p*su+(1-p)*sd)/B1:.4f}   (vs S0 = {S0})')
print(f'E_Q[S1]/B1 = {(q*su+(1-q)*sd)/B1:.4f}   (vs S0 = {S0})')
print(f'E_P[C1]/B1 = {(p*cu+(1-p)*cd)/B1:.4f}   (vs price = {C0_rn:.4f})')


## 4. Method 3 — pricing kernel

A pricing kernel $m>0$ satisfies $A_0=\mathbb{E}_{\mathbb{P}}[m A_1]$ for every asset. In two states $m_\omega=\frac{B_0}{B_1}\frac{q_\omega}{p_\omega}$ (discount $\times$ change of measure).


In [ ]:
m_u = (B0/B1) * (q/p)
m_d = (B0/B1) * ((1-q)/(1-p))
C0_pk = p*m_u*cu + (1-p)*m_d*cd
print(f'm_u = {m_u:.4f}, m_d = {m_d:.4f}')
print(f'C0 (pricing kernel) = {C0_pk:.4f}')
assert np.isclose(B1*(p*m_u + (1-p)*m_d), B0)
assert np.isclose(p*m_u*su + (1-p)*m_d*sd, S0)
print('kernel reprices B and S; note m_d > m_u (payoffs in the bad state cost more)')


All three methods give the **same** price:


In [ ]:
print(round(C0_repl,4), round(C0_rn,4), round(C0_pk,4))
assert np.allclose([C0_repl, C0_rn, C0_pk], C0_repl)


## 5. A reusable one-period pricer


In [ ]:
def one_period_price(S0, B0, B1, su, sd, cu, cd):
    """No-arbitrage price of a one-period claim with payoff (cu, cd)."""
    q = (B1/B0 - sd/S0) / (su/S0 - sd/S0)
    if not (0 < q < 1):
        raise ValueError('arbitrage: q must lie in (0,1)')
    return (B0/B1) * (q*cu + (1-q)*cd)

one_period_price(S0, B0, B1, su, sd, cu, cd)



## 6. Two-period model — backward induction

Now $t=0,1,2$ with $B_0=1,\,B_1=1.05,\,B_2=1.11$ and a recombining stock tree; the claim is a **call with $K=55$**. Each one-period sub-tree is priced as above, working backwards from maturity.


In [ ]:
B = {0: 1.0, 1: 1.05, 2: 1.11}
S = {'u': 115.0, 'd': 75.0, 'uu': 130.0, 'ud': 95.0, 'dd': 45.0}
S0_2, K2 = 100.0, 55.0

Cuu, Cud, Cdd = (max(S[k]-K2, 0.0) for k in ('uu','ud','dd'))
Vu = one_period_price(S['u'], B[1], B[2], S['uu'], S['ud'], Cuu, Cud)
Vd = one_period_price(S['d'], B[1], B[2], S['ud'], S['dd'], Cud, Cdd)
V0 = one_period_price(S0_2, B[0], B[1], S['u'], S['d'], Vu, Vd)
print(f'V_1,u = {Vu:.4f}\nV_1,d = {Vd:.4f}\nV_0   = {V0:.4f}')
assert np.isclose(V0, 51.1583, atol=1e-3)


## 7. General $n$-step binomial and convergence to Black–Scholes

With a CRR parametrisation $u=e^{\sigma\sqrt{\Delta t}},\,d=1/u$ and $\Delta t=T/n$, the $n$-step binomial price of a European call converges to the Black–Scholes price as $n\to\infty$.


In [ ]:
def crr_call(S0, K, r, sigma, T, n):
    dt = T / n
    u = exp(sigma*sqrt(dt)); d = 1/u
    q = (exp(r*dt) - d) / (u - d)
    j = np.arange(n+1)
    ST = S0 * u**j * d**(n-j)
    payoff = np.maximum(ST - K, 0.0)
    weights = np.array([comb(n, k) for k in range(n+1)], dtype=float) * q**j * (1-q)**(n-j)
    return exp(-r*T) * float(np.sum(weights * payoff))

def bs_call(S0, K, r, sigma, T):
    d1 = (log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S0*Phi(d1) - K*exp(-r*T)*Phi(d2)

S0g, Kg, rg, sig, Tg = 100.0, 100.0, 0.03, 0.20, 1.0
ns = np.arange(1, 101)
prices = np.array([crr_call(S0g, Kg, rg, sig, Tg, n) for n in ns])
bs = bs_call(S0g, Kg, rg, sig, Tg)
print(f'Black-Scholes price   = {bs:.4f}')
print(f'binomial price (n=100) = {prices[-1]:.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(ns, prices, lw=1, label='$n$-step binomial')
ax.axhline(bs, color='red', ls='--', label='Black–Scholes')
ax.set_xlabel('number of steps $n$'); ax.set_ylabel('European call price')
ax.set_title('Binomial price converges to Black–Scholes'); ax.legend()
plt.tight_layout(); plt.show()


The even/odd zig-zag is typical of the binomial scheme, but the price clearly settles to the Black–Scholes value as $n$ grows.
